# Harmonic Oscillator Advantage: Stable-Integrator Rerun

**Why this exists:** every Harmonic result in this log so far (Experiment 19,
the n=20 confirmatory rerun, A1's OOD table, A2a) was generated with an
explicit Euler integrator that is unconditionally amplitude-growing for this
oscillator. Directly verified: within a single 512-sample context window, the
signal's std grows 1.37x from first half to second half; across the full
trajectory, early-vs-late window std differs by 42x. This is not the
"stationary sinusoid" the log's framing assumes.

**This notebook reruns the same n=20, H=96/192/336 confirmatory protocol**
with the generator switched to solve_ivp/RK45 (tight tolerances, tracks the
true bounded solution), everything else identical. Also verifies its own
premise first: confirms the RK45 signal does NOT show the growth pathology,
before reporting the main comparison.

**Three-way comparison target:**
- Experiment 19 original: n=8, H=96, Panda advantage = +0.370, p=0.004
- n=20 Euler confirmatory (already logged): H=96, advantage = +0.322, p<0.0001
- This run: n=20, H=96/192/336, RK45 (stable) generator


In [1]:
# ============================================================
# CELL 1 -- IMPORTS + MODEL LOADING (published checkpoints)
# ============================================================
import numpy as np
import pandas as pd
import torch
from scipy.stats import wilcoxon
from scipy.integrate import solve_ivp
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

import sys
sys.path.insert(0, './panda')

from panda.patchtst.pipeline import PatchTSTPipeline
from chronos import ChronosPipeline

panda_model = PatchTSTPipeline.from_pretrained(
    mode="predict",
    pretrain_path="GilpinLab/panda",
    device_map=device,
)

chronos_model = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map=device,
    torch_dtype=torch.bfloat16,
)

print("Models loaded.")


Device: cpu
Models loaded.


In [2]:
# ============================================================
# CELL 2 -- EVALUATION HARNESS (verbatim, unchanged from prior notebooks)
# ============================================================
def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def instance_norm_window(x_CT):
    mu  = x_CT.mean(axis=1, keepdims=True)
    std = x_CT.std( axis=1, keepdims=True) + 1e-8
    return (x_CT - mu) / std, mu, std

CONTEXT_LEN = 512

def panda_forecast(context_np, horizon):
    TRAIN_H   = 128
    remaining = horizon
    ctx       = context_np.copy()
    preds     = []
    while remaining > 0:
        h         = min(TRAIN_H, remaining)
        context_t = torch.tensor(ctx.T, dtype=torch.float32)
        with torch.no_grad():
            pred = panda_model.predict(
                context_t, h,
                limit_prediction_length=False,
                sliding_context=True,
            )
        p = pred.squeeze().cpu().numpy()
        if p.ndim == 1:
            p = p[:, None]
        if p.shape[0] != context_np.shape[0]:
            p = p.T
        preds.append(p[:, :h])
        ctx       = np.concatenate([ctx[:, h:], p[:, :h]], axis=1)
        remaining -= h
    return np.concatenate(preds, axis=1)

def chronos_forecast(context_np, horizon):
    ctx = torch.tensor(context_np, dtype=torch.float32)
    with torch.no_grad():
        out = chronos_model.predict(
            ctx, prediction_length=horizon, num_samples=1
        )
    return out[:, 0, :].cpu().numpy()

def evaluate(data_CT, horizon, n_windows=20, label="",
             fn_a=None, fn_b=None,
             name_a="panda", name_b="chronos"):
    if fn_a is None: fn_a = panda_forecast
    if fn_b is None: fn_b = chronos_forecast

    C, T      = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    if max_start <= 0:
        print(f"  [SKIP] {label}: T={T} too short")
        return None

    starts = np.linspace(0, max_start, n_windows, dtype=int)
    mae_a, mae_b = [], []

    for s in starts:
        ctx_raw = data_CT[:, s : s + CONTEXT_LEN]
        tgt_raw = data_CT[:, s + CONTEXT_LEN : s + CONTEXT_LEN + horizon]
        ctx_norm, mu, std = instance_norm_window(ctx_raw)
        tgt_norm = (tgt_raw - mu) / std

        mae_a.append(mae(tgt_norm, fn_a(ctx_norm, horizon)))
        mae_b.append(mae(tgt_norm, fn_b(ctx_norm, horizon)))

    diff = np.array(mae_b) - np.array(mae_a)
    try:
        if np.any(diff != 0):
            _, pval = wilcoxon(diff, alternative="greater")
        else:
            pval = 1.0
    except Exception:
        pval = np.nan

    adv = np.median(mae_b) - np.median(mae_a)

    result = {
        "label"         : label,
        "horizon"       : horizon,
        f"{name_a}_mae" : np.median(mae_a),
        f"{name_a}_iqr" : np.percentile(mae_a,75)-np.percentile(mae_a,25),
        f"{name_b}_mae" : np.median(mae_b),
        f"{name_b}_iqr" : np.percentile(mae_b,75)-np.percentile(mae_b,25),
        "advantage_mae" : adv,
        "wilcoxon_p"    : pval,
    }
    sig = " *" if pval < 0.05 else (" ~" if pval < 0.10 else "")
    print(
        f"  {label:20s}  H={horizon:4d}  "
        f"panda={np.median(mae_a):.4f}  chronos={np.median(mae_b):.4f}  "
        f"Adv={adv:+.4f}  p={pval:.4f}{sig}"
    )
    return result

print("Helpers defined.")


Helpers defined.


In [3]:
# ============================================================
# CELL 3 -- STABLE HARMONIC GENERATOR (solve_ivp/RK45)
# Same generator built and used in Sweep 1 of the B3a notebook, at the
# baseline omega=1.0. Everything else (n_steps=4000, dt=0.05, seed=42,
# 500-step transient skip) matches the established protocol exactly --
# only the integration method changes.
# ============================================================
SEED = 42
BASE_DT = 0.05
N_STEPS = 4000
TRANSIENT_SKIP = 500

def simulate_harmonic_stable(omega=1.0, dt=BASE_DT, n_steps=N_STEPS, seed=SEED):
    rng = np.random.default_rng(seed)
    def rhs(t, y):
        x, v = y
        return [v, -omega**2 * x]
    ic = [float(rng.standard_normal()), float(rng.standard_normal())]
    sol = solve_ivp(rhs, [0, n_steps*dt], ic,
                     t_eval=np.linspace(0, n_steps*dt, n_steps),
                     method='RK45', rtol=1e-8, atol=1e-8)
    return sol.y[0].astype(np.float32)

def load_harmonic_stable():
    series = simulate_harmonic_stable(omega=1.0, dt=BASE_DT, n_steps=N_STEPS, seed=SEED)
    return series[TRANSIENT_SKIP:][None, :]

data_harmonic_stable = load_harmonic_stable()

# --- Verify the premise: confirm this signal does NOT show the growth pathology ---
early = data_harmonic_stable[0, :512]
late  = data_harmonic_stable[0, -512:]
first_half = early[:256]
second_half = early[256:]
print(f"Shape: {data_harmonic_stable.shape}")
print(f"Early-window std: {early.std():.4f}  |  Late-window std: {late.std():.4f}  "
      f"|  ratio: {late.std()/early.std():.3f}x  (Euler equivalent was ~42x)")
print(f"Within one window, first-half std: {first_half.std():.4f}  |  "
      f"second-half std: {second_half.std():.4f}  |  ratio: {second_half.std()/first_half.std():.3f}x  "
      f"(Euler equivalent was ~1.37x)")
assert late.std() / early.std() < 2.0, \
    "Growth pathology still present -- something is wrong, do not trust results below."
print()
print("Premise verified: this signal is stable. Proceeding to main comparison.")


Shape: (1, 3500)
Early-window std: 0.7600  |  Late-window std: 0.7728  |  ratio: 1.017x  (Euler equivalent was ~42x)
Within one window, first-half std: 0.7606  |  second-half std: 0.7593  |  ratio: 0.998x  (Euler equivalent was ~1.37x)

Premise verified: this signal is stable. Proceeding to main comparison.


In [4]:
# ============================================================
# CELL 4 -- RUN: n_windows=20, H=96/192/336, stable generator
# ============================================================
N_WINDOWS = 20
HORIZONS = [96, 192, 336]

stable_results = []
print("Harmonic oscillator (STABLE generator), n_windows=20:")
print()
for H in HORIZONS:
    r = evaluate(data_harmonic_stable, H, n_windows=N_WINDOWS,
                 label=f"Harmonic_H{H}_stable")
    if r:
        stable_results.append(r)

df_stable = pd.DataFrame(stable_results)
df_stable.to_csv("harmonic_n20_stable_generator_results.csv", index=False)
print()
print("Saved harmonic_n20_stable_generator_results.csv")


Harmonic oscillator (STABLE generator), n_windows=20:



We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Harmonic_H96_stable   H=  96  panda=0.0105  chronos=0.0146  Adv=+0.0040  p=0.0000 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Harmonic_H192_stable  H= 192  panda=0.0243  chronos=0.0178  Adv=-0.0065  p=0.6357


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Harmonic_H336_stable  H= 336  panda=0.0267  chronos=0.0370  Adv=+0.0103  p=0.7147

Saved harmonic_n20_stable_generator_results.csv


In [5]:
# ============================================================
# CELL 5 -- THREE-WAY COMPARISON
# ============================================================
EXP19_ORIGINAL_ADV = 0.370
EXP19_ORIGINAL_P = 0.004
EXP19_ORIGINAL_N = 8

EULER_N20_ADV = 0.322
EULER_N20_P = 0.0000  # reported as <0.0001

stable_h96 = df_stable[df_stable["horizon"] == 96].iloc[0]

print("Comparison at H=96:")
print(f"  Experiment 19 original (n={EXP19_ORIGINAL_N}, EULER):  advantage = {EXP19_ORIGINAL_ADV:+.3f}, p = {EXP19_ORIGINAL_P:.4f}")
print(f"  n=20 confirmatory rerun (EULER, already logged): advantage = {EULER_N20_ADV:+.3f}, p < 0.0001")
print(f"  n=20, THIS RUN (STABLE/RK45):                    advantage = {stable_h96['advantage_mae']:+.3f}, p = {stable_h96['wilcoxon_p']:.4f}")
print()

adv = stable_h96["advantage_mae"]
p = stable_h96["wilcoxon_p"]

if p < 0.05 and adv > 0.5 * EULER_N20_ADV:
    verdict = ("OUTCOME 1: Advantage survives on the stable generator, largely intact. "
               "The Euler growth pathology was NOT the driver of the Harmonic result. "
               "No correction needed to Section 12/18 beyond noting this check was done.")
elif p < 0.05 and 0 < adv <= 0.5 * EULER_N20_ADV:
    verdict = ("OUTCOME 2: Advantage survives but is substantially SMALLER on the stable "
               "generator. Partially artifact-inflated -- Section 12/18 need a caveat "
               "noting the originally-reported magnitude was inflated by the amplitude-"
               "growth pathology, even though the qualitative direction holds.")
else:
    verdict = ("OUTCOME 3: Advantage does not survive (not significant, or reversed) on "
               "the stable generator. The original Harmonic finding was SUBSTANTIALLY an "
               "artifact of testing on an exploding-variance signal, not a genuine "
               "advantage on a stationary periodic system. This requires a real correction "
               "to Section 12's 'chaos is not necessary' headline claim, on the same order "
               "of seriousness as the heterogeneity retraction (Section 12.4).")

print(verdict)
print()
print("Full stable-generator results, all horizons:")
print(df_stable[["label", "horizon", "panda_mae", "chronos_mae", "advantage_mae", "wilcoxon_p"]]
      .round(4).to_string(index=False))


Comparison at H=96:
  Experiment 19 original (n=8, EULER):  advantage = +0.370, p = 0.0040
  n=20 confirmatory rerun (EULER, already logged): advantage = +0.322, p < 0.0001
  n=20, THIS RUN (STABLE/RK45):                    advantage = +0.004, p = 0.0000

OUTCOME 2: Advantage survives but is substantially SMALLER on the stable generator. Partially artifact-inflated -- Section 12/18 need a caveat noting the originally-reported magnitude was inflated by the amplitude-growth pathology, even though the qualitative direction holds.

Full stable-generator results, all horizons:
               label  horizon  panda_mae  chronos_mae  advantage_mae  wilcoxon_p
 Harmonic_H96_stable       96     0.0105       0.0146         0.0040      0.0000
Harmonic_H192_stable      192     0.0243       0.0178        -0.0065      0.6357
Harmonic_H336_stable      336     0.0267       0.0370         0.0103      0.7147
